# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()
print(f"{metadata['name']}: {metadata['description']}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Below, we inspect the available record sets and their fields using the Croissant metadata. All entities are referenced by `@id`.

In [ ]:
from pprint import pprint

# List RecordSet @ids
record_sets = []
if hasattr(dataset.metadata, 'recordSet'):
    if isinstance(dataset.metadata.recordSet, list):
        record_sets = [r['@id'] for r in dataset.metadata.recordSet]
    elif isinstance(dataset.metadata.recordSet, dict):
        record_sets = [dataset.metadata.recordSet['@id']]

# If dataset.metadata.to_json()['recordSet'] exists, use it; otherwise, try loading recordsets directly
if not record_sets:
    # Try to extract recordSets via dataset.record_sets property (if available)
    try:
        record_sets = [rs['@id'] for rs in dataset.record_sets]
    except Exception:
        print('No record sets found in the metadata.')

print('Available Record Sets @ids:')
pprint(record_sets)

# Inspect the columns/fields for each record set
fields_by_recordset = {}
for rs_id in record_sets:
    # Croissant allows you to query schema and fields for a recordset
    try:
        rs_schema = dataset.schema(record_set=rs_id)
        fields = [f['@id'] for f in rs_schema['fields']]
        fields_by_recordset[rs_id] = fields
        print(f'Fields for record set {rs_id}:')
        pprint(fields)
    except Exception as e:
        print(f"Could not load fields for record set {rs_id}: {e}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

We'll loop over each record set and load their records into pandas DataFrames, referencing by their `@id`.

In [ ]:
# Extract data from each record set
dataframes = {}

for record_set in record_sets:
    try:
        records = list(dataset.records(record_set=record_set))
        df = pd.DataFrame(records)
        dataframes[record_set] = df
        print(f"Columns for record set {record_set}:")
        print(df.columns.tolist())
        print(df.head())
    except Exception as e:
        print(f"Could not load records for record set {record_set}: {e}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

For illustration, select one available numeric field from a record set. In this case, suppose Table 1 contains a field for age at diagnosis referenced by its `@id`, e.g. `'age-at-diagnosis'`. Make sure to use exact `@id` from the overview above.

If a numeric field exists (e.g. `age-at-diagnosis`), filter by threshold, normalize the distribution, and optionally group by another field (e.g. `'oligomenorrhea'` or `'infertility'` status, each referenced by its `@id`).

In [ ]:
# Choose record set and field by @id (as available from overview)
example_record_set_id = record_sets[0] if len(record_sets) > 0 else None

# Try known clinical/phenotype fields, using only @id referencing
numeric_field_id = None
group_field_id = None

if example_record_set_id:
    df = dataframes[example_record_set_id]
    # Try to select a numeric field
    for col in df.columns:
        # Heuristically, select columns with numeric values
        if df[col].dtype in ['int64', 'float64']:
            numeric_field_id = col
            break
    # Try to select a group-by field
    for col in df.columns:
        # Heuristically: string or categorical
        if df[col].dtype == 'object' and col != numeric_field_id:
            group_field_id = col
            break

    if numeric_field_id:
        threshold = df[numeric_field_id].mean()
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        if group_field_id and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
            print(f"Grouped data by {group_field_id}:")
            print(grouped_df.head())
    else:
        print("No numeric fields found for EDA.")
else:
    print("No available record set to perform EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Here, we visualize the distribution of the selected numeric field and its relationship to the group field (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if example_record_set_id and numeric_field_id:
    df = dataframes[example_record_set_id]
    plt.figure(figsize=(6,4))
    sns.histplot(df[numeric_field_id], bins=10, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # If group_field is available, visualize relationship
    if group_field_id:
        plt.figure(figsize=(6,4))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()
else:
    print("No numeric field found for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- This notebook demonstrated loading and exploring the FAIR^2 clinical dataset package using the `mlcroissant` library.
- All record sets, fields, and columns were referenced by their `@id` identifiers for clarity and reproducibility.
- Basic EDA and visualization revealed the structure and contents of key clinical phenotypic tables and variant tables.
- For further work, consider deeper phenotype/genotype correlation or clinical analysis, keeping in mind the dataset's limitations (small N, female-only, single-center, rare disease focus).

For more information, refer to the dataset's Croissant schema at:
`https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json`